   "**Step 1 of 5.** Consumes: `benchmarking/instances/QOptLib_0/XSH-n20-k4-01.vrp` (committed). Produces: `experimental-data/instances/toy/toy_{0..4}.{json,mps}` (5 random toy instances) and `experimental-data/instances/qoptlib-permuted/XSH-n20-k4-01_perm{1..10}.{json,mps}` (10 permutations of QOptLib benchmark). Runtime: ~1 minute.\n",

**Note:** These instances are already committed in `experimental-data/instances/`, so this notebook is optional. Run it only if you want to regenerate the instances with different random seeds or parameters.

# Generate VRP Instances for Paper Experiments

This notebook generates two sets of VRP instances used in the paper:

1. **Toy instances**: 5 small random VRPs (5 customers, 3 vehicles, capacity 8) used for the toy HQC-MCMS Benders experiment (Figure 3)
2. **QOptLib permutations**: 10 customer-order permutations of the XSH-n20-k4-01 benchmark instance (20 customers, 4 vehicles) used for the QOptLib MCMS experiment (Figures 1-2)

## 1. Import Libraries

In [1]:
import numpy as np
import json
import math
import os
import pulp
import re

## 2. Generate Toy VRP Instances

Creates 5 random toy VRP instances with:
- 1 depot at coordinates (0,0) to (9,9)
- 5 customers with random demands (1-6)
- 3 vehicles with capacity 8
- Euclidean distance costs

These instances are small enough to solve end-to-end with QAOA-based cut selection in reasonable time (~4h per instance).

In [2]:
# Set seed for reproducibility (same as paper)
np.random.seed(42)

# Output directory
toy_output_dir = "experimental-data/instances/toy"
os.makedirs(toy_output_dir, exist_ok=True)

# Instance parameters
n_toy_instances = 5
n_customers = 5
n_depots = 1
num_vehicles = 3
vehicle_capacity = 8

print(f"Generating {n_toy_instances} toy VRP instances...\n")

for idx in range(n_toy_instances):
    # Random coordinates for depot and customers on 10x10 grid
    coords = np.random.randint(0, 10, size=(n_customers + n_depots, 2), dtype=int)
    
    # Demands: depot is 0, customers random 1-6
    demands = [0] + [int(x) for x in np.random.randint(1, 7, size=n_customers)]
    
    # Euclidean distance matrix
    def euclidean_distance_matrix(coords):
        n = len(coords)
        dist = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                dist[i][j] = round(np.linalg.norm(coords[i] - coords[j]), 2)
        return dist
    
    distance_matrix = euclidean_distance_matrix(coords)
    
    # Create JSON metadata
    vrp_data = {
        "problem_type": "Capacitated VRP",
        "n_customers": n_customers,
        "demands": demands,
        "num_vehicles": num_vehicles,
        "vehicle_capacity": vehicle_capacity,
        "distance_matrix": distance_matrix.tolist(),
        "depots": [0],
        "coordinates": coords.tolist()
    }
    
    # Write JSON
    json_path = os.path.join(toy_output_dir, f"toy_{idx}.json")
    with open(json_path, "w") as f:
        json.dump(vrp_data, f, indent=2)
    
    # Build CVRP MILP model with Miller-Tucker-Zemlin formulation
    n = n_customers + 1  # including depot
    nodes = list(range(n))
    depot = 0
    customers = [i for i in nodes if i != depot]
    
    prob = pulp.LpProblem("CVRP_Toy", pulp.LpMinimize)
    
    # Decision variables
    x = pulp.LpVariable.dicts("x", (nodes, nodes), 0, 1, pulp.LpBinary)
    u = pulp.LpVariable.dicts("u", nodes, 0, vehicle_capacity, pulp.LpContinuous)
    
    # Objective: minimize total distance
    prob += pulp.lpSum(distance_matrix[i][j] * x[i][j] for i in nodes for j in nodes if i != j)
    
    # Each customer visited exactly once
    for j in customers:
        prob += pulp.lpSum(x[i][j] for i in nodes if i != j) == 1
    for i in customers:
        prob += pulp.lpSum(x[i][j] for j in nodes if i != j) == 1
    
    # Flow conservation at depot
    prob += pulp.lpSum(x[depot][j] for j in nodes if j != depot) <= num_vehicles
    prob += pulp.lpSum(x[i][depot] for i in nodes if i != depot) <= num_vehicles
    
    # MTZ subtour elimination + capacity constraints
    for i in customers:
        for j in customers:
            if i != j:
                prob += u[i] - u[j] + vehicle_capacity * x[i][j] <= vehicle_capacity - demands[j]
    
    # Load bounds
    for i in customers:
        prob += u[i] >= demands[i]
        prob += u[i] <= vehicle_capacity
    
    # Write MPS file
    mps_path = os.path.join(toy_output_dir, f"toy_{idx}.mps")
    prob.writeMPS(mps_path)
    
    print(f"  Instance {idx}: {n_customers} customers, demands={demands[1:]}, written to toy_{idx}.{{json,mps}}")

print(f"\nToy instances saved to {toy_output_dir}/")

Generating 5 toy VRP instances...

  Instance 0: 5 customers, demands=[3, 6, 5, 2, 4], written to toy_0.{json,mps}
  Instance 1: 5 customers, demands=[1, 3, 5, 3, 5], written to toy_1.{json,mps}
  Instance 2: 5 customers, demands=[4, 4, 4, 5, 3], written to toy_2.{json,mps}
  Instance 3: 5 customers, demands=[2, 4, 2, 2, 6], written to toy_3.{json,mps}
  Instance 4: 5 customers, demands=[5, 5, 1, 1, 1], written to toy_4.{json,mps}

Toy instances saved to experimental-data/instances/toy/


## 3. Generate QOptLib Permuted Instances

Parses the QOptLib benchmark instance `XSH-n20-k4-01.vrp` (20 customers, 4 vehicles, capacity 100) and generates 10 customer-order variants: `_perm1` keeps the original node order, and `_perm2`...`_perm10` are 9 random permutations (depot fixed) drawn sequentially from a single seeded RNG stream (seed 42), matching the original ad-hoc generation script. This creates 10 structurally equivalent but numerically different MILP instances for robust benchmarking.

In [3]:
# QOptLib VRP parser (CVRPLIB format)
def parse_vrp_file(file_path):
    """Parse a CVRPLIB-format .vrp file."""
    coords = {}
    demands = {}
    depot = []
    dimension = None
    capacity = None
    section = None
    
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            if "DIMENSION" in line:
                dimension = int(line.split(":")[1])
            elif "CAPACITY" in line:
                capacity = int(line.split(":")[1])
            elif line.startswith("NODE_COORD_SECTION"):
                section = "coords"
            elif line.startswith("DEMAND_SECTION"):
                section = "demands"
            elif line.startswith("DEPOT_SECTION"):
                section = "depot"
            elif line.startswith("EOF"):
                break
            else:
                if section == "coords":
                    parts = line.split()
                    idx = int(parts[0])
                    x, y = float(parts[1]), float(parts[2])
                    coords[idx] = (x, y)
                elif section == "demands":
                    parts = line.split()
                    idx = int(parts[0])
                    demand = int(parts[1])
                    demands[idx] = demand
                elif section == "depot":
                    if line != "-1":
                        depot.append(int(line))
    
    return dimension, capacity, coords, demands, depot


def compute_distance_matrix(coords):
    """Compute Euclidean distance matrix from coordinates (rounded to nearest int)."""
    n = len(coords)
    dist = np.zeros((n, n))
    for i in coords:
        for j in coords:
            xi, yi = coords[i]
            xj, yj = coords[j]
            dist[i-1][j-1] = int(round(math.hypot(xi - xj, yi - yj)))
    return dist


def build_cvrp_milp(n, capacity, demands, dist, depot_nodes, num_vehicles=4):
    """Build CVRP MILP model with MTZ formulation."""
    nodes = list(range(1, n + 1))
    customers = [i for i in nodes if i not in depot_nodes]
    
    prob = pulp.LpProblem("CVRP_QOptLib", pulp.LpMinimize)
    
    # Decision variables
    x = pulp.LpVariable.dicts("x", (nodes, nodes), 0, 1, pulp.LpBinary)
    u = pulp.LpVariable.dicts("u", nodes, 0, capacity, pulp.LpContinuous)
    
    # Objective: minimize total distance
    prob += pulp.lpSum(dist[i-1][j-1] * x[i][j] for i in nodes for j in nodes if i != j)
    
    # Each customer visited exactly once
    for j in customers:
        prob += pulp.lpSum(x[i][j] for i in nodes if i != j) == 1
    for i in customers:
        prob += pulp.lpSum(x[i][j] for j in nodes if i != j) == 1
    
    # Flow conservation at depot(s)
    for d in depot_nodes:
        prob += pulp.lpSum(x[d][j] for j in nodes if j != d) >= 1
        prob += pulp.lpSum(x[i][d] for i in nodes if i != d) >= 1
    
    # MTZ subtour elimination + capacity
    for i in customers:
        for j in customers:
            if i != j:
                prob += u[i] - u[j] + capacity * x[i][j] <= capacity - demands[j]
    
    # Load bounds
    for i in customers:
        prob += u[i] >= demands[i]
        prob += u[i] <= capacity
    
    return prob


def export_vrp_json(output_path, n, demands, capacity, dist, depot_nodes, num_vehicles):
    """Export VRP instance as JSON."""
    data = {
        "problem_type": "Capacitated VRP",
        "n_customers": n - len(depot_nodes),
        "demands": [demands[i] for i in sorted(demands)],
        "num_vehicles": num_vehicles,
        "vehicle_capacity": capacity,
        "distance_matrix": dist.tolist(),
        "depots": depot_nodes
    }
    with open(output_path, "w") as f:
        json.dump(data, f, indent=2)

In [4]:
# Parse QOptLib benchmark instance
qoptlib_input = "experimental-data/instances/QOptLib_0/XSH-n20-k4-01.vrp"
qoptlib_output_dir = "experimental-data/instances/qoptlib-permuted"
os.makedirs(qoptlib_output_dir, exist_ok=True)

n, capacity, coords, demands, depot = parse_vrp_file(qoptlib_input)
dist = compute_distance_matrix(coords)
num_vehicles = 4  # From instance name XSH-n20-k4-01

print(f"Parsed QOptLib instance: {n} nodes, {n-1} customers, capacity {capacity}, {num_vehicles} vehicles")
print(f"Generating 10 customer-order variants (perm1 = original order, perm2-10 = 9 random permutations)...\n")

# perm1 keeps the original node order; perm2..perm10 are 9 random permutations drawn
# sequentially from a single np.random.seed(42) reset, matching the original ad-hoc
# generation script (which also reserved perm1 for the unpermuted instance).
np.random.seed(42)
n_variants = 10
node_orders = [list(range(1, n + 1))] + [
    [1] + list(np.random.permutation(range(2, n + 1))) for _ in range(n_variants - 1)
]

for perm_idx, perm in enumerate(node_orders, start=1):
    # Apply permutation to coords and demands
    permuted_coords = {i: coords[perm[i-1]] for i in range(1, n + 1)}
    permuted_demands = {i: demands[perm[i-1]] for i in range(1, n + 1)}
    
    # Recompute distance matrix for permuted coords
    permuted_dist = compute_distance_matrix(permuted_coords)
    
    # Build MILP model
    prob = build_cvrp_milp(n, capacity, permuted_demands, permuted_dist, depot, num_vehicles)
    
    # Export MPS
    mps_path = os.path.join(qoptlib_output_dir, f"XSH-n20-k4-01_perm{perm_idx}.mps")
    prob.writeMPS(mps_path)
    
    # Export JSON
    json_path = os.path.join(qoptlib_output_dir, f"XSH-n20-k4-01_perm{perm_idx}.json")
    export_vrp_json(json_path, n, permuted_demands, capacity, permuted_dist, depot, num_vehicles)
    
    print(f"  Permutation {perm_idx}: written to XSH-n20-k4-01_perm{perm_idx}.{{json,mps}}")

print(f"\nQOptLib permuted instances saved to {qoptlib_output_dir}/")

Parsed QOptLib instance: 21 nodes, 20 customers, capacity 231, 4 vehicles
Generating 10 customer-order variants (perm1 = original order, perm2-10 = 9 random permutations)...

  Permutation 1: written to XSH-n20-k4-01_perm1.{json,mps}
  Permutation 2: written to XSH-n20-k4-01_perm2.{json,mps}
  Permutation 3: written to XSH-n20-k4-01_perm3.{json,mps}
  Permutation 4: written to XSH-n20-k4-01_perm4.{json,mps}


  Permutation 5: written to XSH-n20-k4-01_perm5.{json,mps}


  Permutation 6: written to XSH-n20-k4-01_perm6.{json,mps}
  Permutation 7: written to XSH-n20-k4-01_perm7.{json,mps}
  Permutation 8: written to XSH-n20-k4-01_perm8.{json,mps}
  Permutation 9: written to XSH-n20-k4-01_perm9.{json,mps}
  Permutation 10: written to XSH-n20-k4-01_perm10.{json,mps}

QOptLib permuted instances saved to experimental-data/instances/qoptlib-permuted/


## Done

Generated instances:
- **Toy**: 5 instances in `experimental-data/instances/toy/`
- **QOptLib**: 10 permutations in `experimental-data/instances/qoptlib-permuted/`

These instances are used by notebooks 02 (QOptLib MCMS experiment) and 03 (toy HQC-MCMS experiment).